Movie recommendation system

Referencing https://www.geeksforgeeks.org/machine-learning/recommendation-system-in-python/

GeeksforGeeks is my primary resource for this assignment

1. Setup

In [1]:
import pandas as pd

movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")
tags = pd.read_csv("data/tags.csv")
links = pd.read_csv("data/links.csv")

Initial setup and reading of datasets. I'm including all datasets for clarity purpose but I will be moving forward with movies and ratings. I think these will be best for collaborative filtering which is the basis I want to use for my recommender system. Tags and links provide additional information but it is less user behavior-based and more auxiliary.

2. Combining Movies + Ratings

To further explain, the rationale here is movies+ratings are the two most relevant datasets for this recommendation system. The reason being is tags and links are both more optional. They both fall into external filtering or websites as opposed to direct reviews and ratings by consumers. I want my recommendation system to focus more on the actual user experience instead of overindexing on 3rd party sources or grouping by tags.

In [2]:
#merging ratings with movie info
movie_ratings = ratings.merge(movies, on="movieId")

#preview
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,17,1,4.5,1305696483,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy


3. Movie-User Matrix

In [3]:
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

def create_matrix(df):
    
    #numeric index for each user
    user_mapper = {user: i for i, user in enumerate(df["userId"].unique())}
    
    #numeric index for each movie
    movie_mapper = {movie: i for i, movie in enumerate(df["movieId"].unique())}
    
    #reverse look-up to allow row number conversion into movieIDs
    movie_inv_mapper = {i: movie for movie, i in movie_mapper.items()}

    
    #mapping userIDs and movieIDs to matrix positions
    user_index = df["userId"].map(user_mapper)
    movie_index = df["movieId"].map(movie_mapper)

    #matrix with rows as movies, users as columns, values as ratings
    X = csr_matrix(
        (df["rating"], (movie_index, user_index)),
        shape=(len(movie_mapper), len(user_mapper))
    )

    return X, movie_mapper, movie_inv_mapper

I'm creating the recommendation matrix. I'm using ratings and also referencing the GeeksforGeeks example. I'm implementing collaborative filtering which means instead of prioritizing by genre or description, I am comparing movies based on user ratings which in my mind is a better reflection of actual user behavior. This is in line with my initial thinking when I chose to exclude tags/links from my system. I want the recommender to emphasize user ratings and find movies that fall into those patterns of review.

4. Matrix

In [4]:
X, movie_mapper, movie_inv_mapper = create_matrix(movie_ratings)

X.shape

(9724, 610)

The matrix is 9724 by 610, meaning there are 9724 movies being compared across 610 users.

5. Dictionaries for Title Lookup

In [5]:
#dict converting movie title to movieID
movie_to_id = dict(zip(movies["title"], movies["movieId"]))

#dict converting movieID back to movie title
id_to_movie = dict(zip(movies["movieId"], movies["title"]))

This allows the recommender system to take a 'movie title' input, convert it to a 'movie ID' for lookup, and then it's converted back to a normal movie title as output for the user.

6. Fitting Nearest-Neighbors Model

In [6]:
model = NearestNeighbors(
    metric="cosine", #cosine to compare movies based on rating patterns instead of raw value
    algorithm="brute", #search all movies to find nearest neighbors
    n_neighbors=11 #return 11 results because first result is the initial movie
)

#fitting model to the matrix
model.fit(X)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=11)

I'm fitting the nearest neighbors model to the earlier movie-user matrix. I am opting for cosine distance to measure rating patterns vs raw values. This is better for collaborative filtering because it identifies movies that receive similar user rating behavior. In other words, the recommendation system will find movies that have similar viewing patterns which in turn should allow it to generate better recommendations.

7. Recommendation Function

In [7]:
def recommend_movies(movie_title,
                     model,
                     X,
                     movie_to_id,
                     movie_mapper,
                     movie_inv_mapper,
                     id_to_movie,
                     n_recommendations=10):

    #checking for existence of movie title
    #otherwise, error message given
    if movie_title not in movie_to_id:
        print("Movie not found. Check spelling and include the year.")
        return

    #convert movie title to movieID
    movie_id = movie_to_id[movie_title]
    
    #convert movieID to row position in matrix
    movie_index = movie_mapper[movie_id]

    #use cosine distance to find nearest movies
    distances, indices = model.kneighbors(
        X[movie_index],
        n_neighbors=n_recommendations + 1
    )

    print(f"Because you liked: {movie_title}")
    print("\nRecommended movies:")

    #skips initial result because it's the movie the search is based on
    for i, idx in enumerate(indices.flatten()[1:], start=1):
        recommended_movie_id = movie_inv_mapper[idx]
        recommended_title = id_to_movie[recommended_movie_id]

        print(f"{i}. {recommended_title}")

8. Testing Recommender

In [8]:
recommend_movies(
    "Toy Story (1995)",
    model,
    X,
    movie_to_id,
    movie_mapper,
    movie_inv_mapper,
    id_to_movie
)

Because you liked: Toy Story (1995)

Recommended movies:
1. Toy Story 2 (1999)
2. Jurassic Park (1993)
3. Independence Day (a.k.a. ID4) (1996)
4. Star Wars: Episode IV - A New Hope (1977)
5. Forrest Gump (1994)
6. Lion King, The (1994)
7. Star Wars: Episode VI - Return of the Jedi (1983)
8. Mission: Impossible (1996)
9. Groundhog Day (1993)
10. Back to the Future (1985)


This initial test is successful. Based on 'Toy Story,' the recommender returned movies that make sense given the time period and user rating patterns. Movies like the sequel 'Toy Story 2,' Jurassic Park, Star Wars, Lion King: all of these were very popular in the 90s and are adventure-style films that families could watch together at home or in theatres.

9. Secondary Test

In [9]:
recommend_movies(
    "Jumanji (1995)",
    model,
    X,
    movie_to_id,
    movie_mapper,
    movie_inv_mapper,
    id_to_movie
)

Because you liked: Jumanji (1995)

Recommended movies:
1. Lion King, The (1994)
2. Mrs. Doubtfire (1993)
3. Mask, The (1994)
4. Jurassic Park (1993)
5. Home Alone (1990)
6. Nightmare Before Christmas, The (1993)
7. Aladdin (1992)
8. Beauty and the Beast (1991)
9. Ace Ventura: When Nature Calls (1995)
10. Santa Clause, The (1994)


This second test is also successful. Using Jumanji, the recommender provided results that make sense given user rating patterns. Robin Williams is in Jumanji, and he is also in Mrs. Doubtfire. The other recommended movies have famous comedy actors as well: Mask and Ace Ventura both have Jim Carrie, Santa Clause has Tim Allen. While the recommender doesn't use actor metadata, it uses user ratings which means crossover across comedy and adventure is being seen here. Jurassic Park is another adventure film popular in the 90s with a similar jungle vibe to Jumanji. Given the context of the output, the recommender system output makes sense.

10. Final Input / Output for User

In [ ]:
#user prompt for movie title input
user_movie = input(
    "Enter a movie title (example: Toy Story (1995)): "
)

recommend_movies(
    user_movie,
    model,
    X,
    movie_to_id,
    movie_mapper,
    movie_inv_mapper,
    id_to_movie
)

11. Conclusion

In conclusion, my recommender system based on GeeksforGeeks is successful in using collaborative filtering. It recommends movies based on user rating patterns instead of raw user score, genre or tags. This means the system compares how users rate movies across the dataset and in turn, identifies movies with similar ratings based on audience behavior. The output is in line with this thinking, suggesting the model is performing to an expected level for movie recommendations. To improve and check for further consistency, I would test movies in completely different genres and further down the list. I used the 1st and 2nd movies choices from the csv, but I'm curious to see how a more obscure movie would perform in the recommender system.